In [ ]:
!pip install -q wikipedia
import wikipedia

  Preparing metadata (setup.py) ... done


In [ ]:
wikipedia.set_user_agent("MyKaggleBot/1.0 (your_email@example.com)")

In [ ]:
import pandas as pd
topic_df = pd.read_csv("/content/wiki_topics.csv", encoding="utf-8")

In [ ]:
topic_df

,id,topics
0,0,"['Supersymmetric quantum mechanics', 'Hamilton..."
1,1,"['CEERS-93316', 'Redshift', 'James Webb Space ..."
2,2,"['Sun', 'Atmosphere of Earth', 'Rayleigh scatt..."
3,3,"['Redshift', 'Cosmic expansion', 'Universe']"
4,4,"['Landau-Lifshitz-Gilbert equation', 'Magnetiz..."
...,...,...
495,495,"['Cold dark matter', 'Weakly interacting massi..."
496,496,"['Planetary system', 'Solar System', 'Gravitat..."
497,497,"['Magnetic susceptibility', 'Magnetization', '..."
498,498,"['Sagittarius A*', 'Supermassive black hole', ..."


In [ ]:
from tqdm.notebook import tqdm
import time
import ast
documents = []
topics_list = []
missing_topics = []
for topics in topic_df.iloc[:, 1]:
    topics_list.extend(ast.literal_eval(topics))

topics_list = sorted(set(topics_list))

for topic in tqdm(topics_list, desc="Running..."):
  try:
    page = wikipedia.page(topic, auto_suggest=False)
    documents.append({
        "title": page.title,
        "content": page.content
    })
    time.sleep(0.5)
  except wikipedia.DisambiguationError as e:
    page = wikipedia.page(e.options[0], auto_suggest=False)
    documents.append({
        "title": page.title,
        "content": page.content
    })
    time.sleep(0.5)
  except Exception as e:
    print(f"Topic Error: {topic}: {e}")
    missing_topics.append(topic)
    continue

len(documents)

Running...:   0%|          | 0/640 [00:00<?, ?it/s]

Topic Error: AT-cut crystal: Page id "AT-cut crystal" does not match any pages. Try another id!
Topic Error: Aerosol physics: Page id "Aerosol physics" does not match any pages. Try another id!
Topic Error: Atomristor: Page id "Atomristor" does not match any pages. Try another id!
Topic Error: Biological structure: Page id "Biological structure" does not match any pages. Try another id!


/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Topic Error: CYCLOIDEA: Page id "CYCLOIDEA" does not match any pages. Try another id!
Topic Error: Cosmological distance measures: Page id "Cosmological distance measures" does not match any pages. Try another id!
Topic Error: Cryogenic detector: Page id "Cryogenic detector" does not match any pages. Try another id!
Topic Error: Electromagnetic absorption: Page id "Electromagnetic absorption" does not match any pages. Try another id!
Topic Error: Ferroelectric domain: Page id "Ferroelectric domain" does not match any pages. Try another id!
Topic Error: Fibrous rings of the heart: Page id "Fibrous rings of the heart" does not match any pages. Try another id!
Topic Error: Grain size (metallurgy): Page id "Grain size (metallurgy)" does not match any pages. Try another id!
Topic Error: Liouville's theorem (statistical mechanics): Page id "Liouville's theorem (statistical mechanics)" does not match any pages. Try another id!
Topic Error: Maximal acceleration: Page id "Maximal acceleration" 

612

In [ ]:
print(len(missing_topics))
missing_topics

28


['AT-cut crystal',
 'Aerosol physics',
 'Atomristor',
 'Biological structure',
 'CYCLOIDEA',
 'Cosmological distance measures',
 'Cryogenic detector',
 'Electromagnetic absorption',
 'Ferroelectric domain',
 'Fibrous rings of the heart',
 'Grain size (metallurgy)',
 "Liouville's theorem (statistical mechanics)",
 'Maximal acceleration',
 'Mechanical properties of materials',
 'Noble liquid detector',
 'Non-baryonic dark matter',
 'Non-volatile matter',
 'Permutation-inversion group',
 'Pressure differential',
 'Proeutectoid',
 'Proximity focusing',
 'Quark–diquark model',
 'Rectilinear propagation of light',
 'Resistance switching',
 "Roemer's determination of the speed of light",
 'Statistical fluid mechanics',
 'Ultra-low field MRI',
 'Wigner quasiparticle distribution']

In [ ]:
def fetch_page(topic):
  try:
    return wikipedia.page(topic, auto_suggest = True)
  except:
    pass
  print("No Page found")
  results = wikipedia.search(topic)
  if results:
    try:
      return wikipedia.page(results[0], auto_suggest=False)
    except:
      pass
  print("No page found by search")

  return None

In [ ]:
final_list_missing = []
for topic in missing_topics:
  page = fetch_page(topic)

  if page is None:
    final_list_missing.append(topic)
  else:
    print(f"Appending {page.title}")
    documents.append({
        "title": page.title,
        "content": page.content
    })
    time.sleep(0.5)
print(final_list_missing)

Appending Crystal oscillator
Appending Deposition (aerosol physics)
Appending Memristor
Appending Structure
Appending Cyclida
Appending Distance measure
Appending Cryogenic particle detector
Appending Absorption (electromagnetic radiation)
Appending Ferroelectricity
Appending Cardiac skeleton
Appending Grain boundary strengthening
Appending Liouville's theorem (Hamiltonian)
No Page found
Appending Accelerationism
Appending List of materials properties
Appending Dark matter
Appending Baryonic dark matter
Appending Volatile organic compound
Appending Inversion (discrete mathematics)
Appending Pressure measurement
Appending Pearlite
Appending Cherenkov radiation
Appending Quark model
Appending Rectilinear propagation
Appending Memristor
Appending Rømer's determination of the speed of light
Appending Fluid mechanics
Appending Magnetic resonance imaging
Appending Quantum field theory
[]


In [ ]:
len(documents)

640

In [ ]:
output_df = pd.DataFrame(documents)
output_df.to_csv('Wiki_knowledge_raw_updated.csv', index=False, encoding="utf-8")